In [1]:
import os
import gc
import csv
import psutil
import ctypes
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingLR
from torch.optim import Adam, AdamW
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torchvision
from torch.utils.data import TensorDataset, DataLoader

In [2]:
class FeatureExtractor(nn.Module):
    """Pretrained backbone"""
    def __init__(self, feat_dim=256):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

        old_weight = base.conv1.weight
        new_conv = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
        new_conv.weight.data = old_weight.mean(dim=1, keepdim=True)
        base.conv1 = new_conv

        self.backbone = nn.Sequential(
            base.conv1,
            base.bn1,
            base.relu,
            base.maxpool,
            base.layer1,
            base.layer2,
            base.layer3,
            base.layer4,
        )
        self.pool = base.avgpool

        self.bottleneck = nn.Sequential(
            nn.Linear(512, feat_dim),
            nn.BatchNorm1d(feat_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2)
        )
        self.feat_dim = feat_dim

    def forward(self, x):
        x = self.backbone(x)
        x = self.pool(x).flatten(1)
        x = self.bottleneck(x)
        return x


class Classifier(nn.Module):
    """Classification head"""
    def __init__(self, feat_dim=256, num_classes=4):
        super().__init__()
        self.fc = nn.Linear(feat_dim, num_classes)
        self.dropout = nn.Dropout(p=0.2)

    def forward(self, feat):
        feat = self.dropout(feat)
        return self.fc(feat)


class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.clone()

    @staticmethod
    def backward(ctx, grad):
        return -ctx.alpha * grad, None

def gradient_reverse(x, alpha=1.0):
    return GradientReversal.apply(x, alpha)

In [3]:
class DomainDiscriminator(nn.Module):
    THRESHOLD = 2048
    
    def __init__(self, feat_dim=256, num_classes=4, hidden=512, rand_dim=1024):
        super().__init__()
        raw_dim = feat_dim * num_classes
        self.use_random = raw_dim > self.THRESHOLD
        self.rand_dim = rand_dim

        if self.use_random:
            self.Rf = nn.Parameter(torch.empty(rand_dim, feat_dim), requires_grad=False)
            self.Rg = nn.Parameter(torch.empty(rand_dim, num_classes), requires_grad=False)
            nn.init.uniform_(self.Rf, -(3**0.5), 3**0.5)
            nn.init.uniform_(self.Rg, -(3**0.5), 3**0.5)
            in_dim = rand_dim
        else:
            in_dim = raw_dim

        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(hidden // 2, 1),
        )

    def _multilinear_map(self, feat, prob):
        if self.use_random:
            rf = feat @ self.Rf.T
            rg = prob @ self.Rg.T
            return (rf * rg) / (self.rand_dim ** 0.5)
        else:
            joint = torch.bmm(feat.unsqueeze(2), prob.unsqueeze(1))
            return joint.flatten(1)

    def forward(self, feat, logits, alpha=1.0):
        prob = torch.softmax(logits, dim=1)
        joint = self._multilinear_map(feat, prob)
        joint = torch.nn.functional.normalize(joint, dim=1)
        joint = gradient_reverse(joint, alpha)
        return self.net(joint)

In [4]:
class CDAN(nn.Module):
    def __init__(self, num_classes=4, feat_dim=256):
        super().__init__()
        self.extractor = FeatureExtractor(feat_dim)
        self.classifier = Classifier(feat_dim, num_classes)
        self.discriminator = DomainDiscriminator(feat_dim, num_classes)

    def forward(self, x, alpha=1.0):
        feat = self.extractor(x)
        logits = self.classifier(feat)
        domain_logits = self.discriminator(feat, logits, alpha)
        return logits, domain_logits, feat

def get_alpha(epoch, max_epochs, gamma=10.0):
    p = epoch / max_epochs
    return 2.0 / (1.0 + np.exp(-gamma * p)) - 1.0

In [ ]:
class OW_MMD(nn.Module):
    def __init__(self, kernel_num=5, kernel_mul=2.0, lambda_reg=1e-3):
        super().__init__()
        self.kernel_num = kernel_num
        self.kernel_mul = kernel_mul
        self.lambda_reg = lambda_reg

    def gaussian_kernels(self, src, tgt):
        total = torch.cat([src, tgt], dim=0)

        norm = (total**2).sum(dim=1, keepdim=True)
        D = norm + norm.T - 2 * total @ total.T
        D = torch.clamp(D, min=0)

        bandwidth = torch.median(D.detach())
        if bandwidth < 1e-6:
            bandwidth = torch.tensor(1.0, device=src.device)
        bandwidth = bandwidth / (self.kernel_mul ** (self.kernel_num // 2))

        bandwidth_list = [bandwidth * (self.kernel_mul ** i) for i in range(self.kernel_num)]
        kernels = [torch.exp(-D / (2 * bw)) for bw in bandwidth_list]

        return kernels

    def forward(self, src_feat, tgt_feat):
        B = src_feat.size(0)

        kernels = self.gaussian_kernels(src_feat, tgt_feat)
        K = sum(kernels) / len(kernels)

        K_ss = K[:B, :B]
        K_st = K[:B, B:]
        K_tt = K[B:, B:]

        z = torch.ones(B, device=K.device) / B

        K_ss_reg = K_ss + self.lambda_reg * torch.eye(B, device=K.device)
        w = torch.linalg.solve(K_ss_reg, z)

        SS = (w.unsqueeze(0) @ K_ss @ w.unsqueeze(1)).squeeze()
        ST = (w @ K_st).mean()
        TT = K_tt.mean()

        loss = SS - 2 * ST + TT
        return torch.clamp(loss, min=0)

In [39]:
class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device
        N, V, D = features.shape

        features = F.normalize(features, dim=2)
        features = features.view(N * V, D)

        sim = torch.matmul(features, features.T) / self.temperature

        mask_eye = torch.eye(N * V, dtype=torch.bool, device=device)
        sim = sim.masked_fill(mask_eye, -1e9)

        labels_rep = labels.repeat_interleave(V).view(-1, 1)
        mask = torch.eq(labels_rep, labels_rep.T).float().to(device)

        exp_sim = torch.exp(sim)
        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True))

        mask.fill_diagonal_(0)

        mean_log_prob_pos = (mask * log_prob).sum(1) / mask.sum(1).clamp(min=1)

        loss = -mean_log_prob_pos
        return loss.view(N, V).mean()

In [ ]:
class Trainer:
    def __init__(self, model: CDAN, src_loader: DataLoader, tgt_loader: DataLoader, num_classes: int=4,
                lr: float=1e-4, max_epochs: int=100, device: str="cuda", lambda_mmd: float=0.3, lambda_supcon: float=0.3, save_path: str="CDAN_BEST.pth"):

        self.model = model.to(device)
        self.src_loader = src_loader
        self.tgt_loader = tgt_loader
        self.max_epochs = max_epochs
        self.device = device
        self.lambda_mmd = lambda_mmd
        self.lambda_supcon = lambda_supcon
        self.save_path = save_path
    
        backbone_params = list(model.extractor.backbone.parameters())
        new_params = (
            list(model.extractor.bottleneck.parameters()) +
            list(model.classifier.parameters())
        )
        disciminator_params = list(model.discriminator.parameters())
    
        self.optimizer = AdamW(
            [{"params": backbone_params, "lr": lr * 0.1},
            {"params": new_params, "lr": lr}],
            weight_decay=1e-4
        )
        self.disc_optimizer = AdamW(disciminator_params, lr=lr * 3)

        steps_per_epoch = len(src_loader)

        self.scheduler = OneCycleLR(
            self.optimizer,
            max_lr=[group["lr"] for group in self.optimizer.param_groups],
            steps_per_epoch=steps_per_epoch,
            epochs=max_epochs,
            pct_start=0.1,
            div_factor=25.0,
            final_div_factor=1e4,
            anneal_strategy="cos",
        )

        self.disc_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.disc_optimizer,
            T_max=max_epochs,
            eta_min=lr * 0.3,
        )

        self.cls_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
        self.disc_loss_fn = nn.BCEWithLogitsLoss(reduction="none")
        self.mmd_loss_fn = OW_MMD(kernel_num=5).to(device)
        self.supcon_loss_fn = SupConLoss(temperature=0.7).to(device)

        self.history = {"cls loss": [], "disc loss": [], "mmd loss": [], "supcon loss": [], "total loss": []}
        self.best_cls_loss = float("inf")

        self.csv_path = save_path.replace(".pth", "_history.csv")
        with open(self.csv_path, "w", newline="") as f:
            csv.writer(f).writerow([
                "epoch", "alpha", "cls_loss", "disc_loss", "mmd_loss", "supcon_loss", "total_loss",
            ])

        self._libc = ctypes.CDLL("libc.so.6")

    def _train_epoch(self, epoch):
        self.model.train()
        alpha = get_alpha(epoch, self.max_epochs)
        lambda_adv = alpha

        src_label = lambda n: torch.zeros(n, 1, device=self.device)
        tgt_label = lambda n: torch.ones(n, 1, device=self.device)

        total_cls = total_disc = total_mmd = total_supcon = total = 0.0
        steps = 0
        tgt_iter = iter(self.tgt_loader)

        for src_imgs, src_y in self.src_loader:
            try:
                tgt_imgs, _ = next(tgt_iter)
            except StopIteration:
                tgt_iter = iter(self.tgt_loader)
                tgt_imgs, _ = next(tgt_iter)
                
            src_imgs = src_imgs.to(self.device, non_blocking=True)
            src_y = src_y.to(self.device, non_blocking=True)
            tgt_imgs = tgt_imgs.to(self.device, non_blocking=True)

            bs = min(src_imgs.size(0), tgt_imgs.size(0))

            with torch.no_grad():  # Update Discriminator
                src_logits_d, _, src_feat = self.model(src_imgs, alpha)
                tgt_logits_d, _, tgt_feat = self.model(tgt_imgs, alpha)

                src_prob_d = F.softmax(src_logits_d[:bs], dim=1)
                tgt_prob_d = F.softmax(tgt_logits_d[:bs], dim=1)

                src_entropy_d = -(src_prob_d * torch.log(src_prob_d + 1e-8)).sum(dim=1)
                tgt_entropy_d = -(tgt_prob_d * torch.log(tgt_prob_d + 1e-8)).sum(dim=1)

                src_w_d = 1.0 + torch.exp(-src_entropy_d)
                tgt_w_d = 1.0 + torch.exp(-tgt_entropy_d)

                src_w_d = src_w_d / (src_w_d.sum().detach() + 1e-8)
                tgt_w_d = tgt_w_d / (tgt_w_d.sum().detach() + 1e-8)

            src_d_only = self.model.discriminator(src_feat.detach(), src_logits_d.detach(), alpha)
            tgt_d_only = self.model.discriminator(tgt_feat.detach(), tgt_logits_d.detach(), alpha)

            src_loss_d = self.disc_loss_fn(src_d_only[:bs], src_label(bs)).view(-1)
            tgt_loss_d = self.disc_loss_fn(tgt_d_only[:bs], tgt_label(bs)).view(-1)

            disc_loss = ((src_w_d * src_loss_d).sum()  + (tgt_w_d * tgt_loss_d).sum())  * 0.5

            self.disc_optimizer.zero_grad(set_to_none=True)
            disc_loss.backward()
            self.disc_optimizer.step()

            src_logits, src_d, src_feat = self.model(src_imgs, alpha)
            tgt_logits, tgt_d, tgt_feat = self.model(tgt_imgs, alpha)

            cls_loss = self.cls_loss_fn(src_logits, src_y)

            with torch.no_grad():  # Update backbone and classifier
                src_prob = F.softmax(src_logits[:bs], dim=1)
                tgt_prob = F.softmax(tgt_logits[:bs], dim=1)

                src_entropy = -(src_prob * torch.log(src_prob + 1e-8)).sum(dim=1)
                tgt_entropy = -(tgt_prob * torch.log(tgt_prob + 1e-8)).sum(dim=1)

                src_w = 1.0 + torch.exp(-src_entropy)
                tgt_w = 1.0 + torch.exp(-tgt_entropy)

                src_w = src_w / (src_w.sum().detach() + 1e-8)
                tgt_w = tgt_w / (tgt_w.sum().detach() + 1e-8)

            src_loss = self.disc_loss_fn(src_d[:bs], src_label(bs)).view(-1)
            tgt_loss = self.disc_loss_fn(tgt_d[:bs], tgt_label(bs)).view(-1)

            disc_loss = ((src_w * src_loss).sum()  + (tgt_w * tgt_loss).sum())  * 0.5
            mmd_loss = self.mmd_loss_fn(src_feat[:bs], tgt_feat[:bs])
            supcon_loss = self.supcon_loss_fn(src_feat[:bs].unsqueeze(1), src_y[:bs])

            loss = cls_loss + lambda_adv * disc_loss + self.lambda_mmd * mmd_loss + self.lambda_supcon * supcon_loss
            self.optimizer.zero_grad(set_to_none=True)
            loss.backward()
            self.optimizer.step()
            self.scheduler.step()

            total_cls += cls_loss.item()
            total_disc += disc_loss.item()
            total_mmd += mmd_loss.item()
            total_supcon += supcon_loss.item()
            total += loss.item()
            steps += 1

            self._libc.malloc_trim(0)

        n = max(steps, 1)
        return total_cls / n, total_disc / n, total_mmd / n, total_supcon / n, total / n

    def train(self):
        print(f"Training CDAN for {self.max_epochs} epochs on {self.device}")
        for epoch in range(1, self.max_epochs + 1):
            cls_loss, disc_loss, mmd_loss, supcon_loss, total_loss = self._train_epoch(epoch)
            alpha = get_alpha(epoch, self.max_epochs)

            self.history['cls loss'].append(float(cls_loss))
            self.history['disc loss'].append(float(disc_loss))
            self.history['mmd loss'].append(float(mmd_loss))
            self.history['total loss'].append(float(total_loss))
            self.history['supcon loss'].append(float(supcon_loss))

            lrs = [pg["lr"] for pg in self.optimizer.param_groups]
            disc_lr = self.disc_optimizer.param_groups[0]["lr"]

            print(
                f"\nEpoch {epoch:03d}/{self.max_epochs} | α={alpha:.3f} |"
                f"lr={lrs[0]:.2e}/{lrs[1]:.2e} | disc_lr={disc_lr:.2e}\n"
                f"  Loss: cls={cls_loss:.4f} | disc={disc_loss:.4f} | mmd={mmd_loss:.4f} | supcon={supcon_loss:.4f} | total={total_loss:.4f}\n"
            )

            with open(self.csv_path, "a", newline="") as f:
                csv.writer(f).writerow([
                    epoch, f"{alpha:.4f}",
                    f"{cls_loss:.6f}", f"{disc_loss:.6f}", f"{mmd_loss:.6f}", f"{supcon_loss:.6f}", f"{total_loss:.6f}",
                ])

            if cls_loss < self.best_cls_loss:
                self.best_cls_loss = cls_loss
                torch.save(self.model.state_dict(), self.save_path)
                print(f"  Saved best model: cls={cls_loss:.4f}")

            self.disc_scheduler.step()
            torch.cuda.empty_cache()
            gc.collect()
            self._libc.malloc_trim(0)

        print("Training complete.")
        return self.history